### Tools

models can request to call tasks such as fetching data from a database , searching the web , or running code . Tools are pairings of :  a schema  including name of the tool , a description , and/or argument definitions(often a JSON schema) and a function / coroutine to execute .

In [8]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 1. Load environment variables (Make sure GEMINI_API_KEY is in your .env)
load_dotenv()

# 2. Initialize the Gemini 2.5 Flash model
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

# 3. Invoke the model
response = model.invoke("write me a 200 words paragraph on Artificial Intelligence")
print(response.content)

Artificial Intelligence (AI) represents the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, problem-solving, perception, and language comprehension. Powered by vast datasets and sophisticated algorithms, AI is no longer a futuristic concept but an integral part of our daily lives.

From personal assistants like Siri and Alexa to recommendation engines on streaming platforms, and from advanced medical diagnostics to self-driving vehicles, AI's applications are diverse and transformative. It promises unprecedented efficiency, innovation across industries, and solutions to complex global challenges, revolutionizing how we work, live, and interact with technology. However, its rapid evolution also brings critical considerations regarding ethics, privacy, bias, and the future of work. As AI continues to advance, understanding its potential and navigating its challenges will be paramount to shaping a future whe

In [10]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """ Get the weather at a location """
    return f"It's sunny in{location}"

model_with_tools=model.bind_tools([get_weather])

In [12]:

response = model_with_tools.invoke("whats the weather in nyc")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "nyc"}'}, '__gemini_function_call_thought_signatures__': {'87ffb525-5e4a-47a9-9504-4590cba2fbd5': 'CvABAQw51scT4H3cHK51/jZsKtzHrOl/MWLTuyop8x++6zG4xoLlhG/hqcirltfY2z0hd6sbVIwloYxt9OZW4l0Kt5rEwJQWqEn3UW3IARUykDIsBeHjsdTvOf11fNbvv9wElutNSDFPhLb/hXwE3ale1SQOnpG1Av5ah+eWNGQOPkr+YQ/vae67bujH5ilIgLFSlbkQLS3MYv/V1fvCTeMG1D0orJU9j4bQOJZBCnbOFLT+9B9NAyYFdLL0lRrNSYvGrwpnshdOaSj2jgdkWIjffrarpcSxxYOcvY0NYKDvAd5y+KUjqfkkh6AjFKR95L4E'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019ed9a4-df1c-7a00-a35e-c26bc76f8467-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'nyc'}, 'id': '87ffb525-5e4a-47a9-9504-4590cba2fbd5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 44, 'output_tokens': 65, 'total_tokens': 109, 'input_token_details': {'cache_read': 0}, 'output_token_

In [14]:
messages = [{"role":"user" , "content": "whats the weather in boston"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Changed ai_msg.tools_calls -> ai_msg.tool_calls
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.


In [15]:
messages

[{'role': 'user', 'content': 'whats the weather in boston'},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "boston"}'}, '__gemini_function_call_thought_signatures__': {'53a977cf-c03b-400f-a4aa-c40fdb54616f': 'CukBAQw51sd2hyLTr8JGzCLzaRs2gUZ/SfBlPaAXZ6mEWm+NneKbcV49qAqHr0buyoLfEmSUJQllkwWHLuXE4VN75uoGM4+kIVMir4YNmUvqdltEHCyZ5dXZ1iux4d9XCA8TcFGBwtcg8OoF/xQB+JFP1OVc+LrhUyqwRUfVon/NLuuSZiofyEbZVk93q0lMu2xhrjjNMjRkKrypekQq0wvKfCFd5+3ngtZvhx1Wj3sJd04qPdGDuVksewIu74/yVptNDlwNqCPSmhtKL5Yg1bc8mF2HQKHmHl+S5I+kDAWWp67zkNWCzddlXds='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ed9aa-a70a-7350-9ef9-72f4db1455de-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'boston'}, 'id': '53a977cf-c03b-400f-a4aa-c40fdb54616f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 61, '